# Setup

In [1]:
# setup module search path
import sys
import importlib
import torch

sys.path.append('./src')

def reload_src():
    for _, mod in list(sys.modules.items()):
        if hasattr(mod, '__file__') and mod.__file__ and '/src/' in mod.__file__:
            importlib.reload(mod)

In [2]:
LOGS_DIR = './logs'
MODELS_DIR = './models'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'{device=}')

def exp_id(postfix: str) -> str:
    return f'exp_{postfix}'

device=device(type='cpu')


# 1

In [3]:
import part_1 as p1

EPOCHS_1 = 500
HIDDEN_SIZE_1 = 32
LR_1 = 1e-3

def exp_id_1(subpart: str) -> str:
    return exp_id(f'1_{subpart}')

## 1.1

Train a 2-hidden-layer MLP on these features to predict $T_c$

In [4]:
HIDDEN_SIZES_1_1 = [HIDDEN_SIZE_1] * 2 # 2 hidden layers
ACTIVATION_1_1 = 'relu'
OPTIMIZER_1_1 = 'sgd'

In [5]:
reload_src()

p1.run_1(
    p1.RunConfig(
        exp_id=exp_id_1('1'),
        device=device,
        output=p1.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p1.Hyperparams(
            epochs=EPOCHS_1,
            lr=LR_1,
            momentum=0.0,
            hidden_sizes=HIDDEN_SIZES_1_1,
            activation=ACTIVATION_1_1,
            optimizer=OPTIMIZER_1_1,
        ),
    )
)

epoch: 0, train_loss: 1.0075, val_loss: 1.0176
epoch: 1, train_loss: 0.9948, val_loss: 1.0108
epoch: 2, train_loss: 0.9889, val_loss: 1.0072
epoch: 3, train_loss: 0.9848, val_loss: 1.0031
epoch: 4, train_loss: 0.9812, val_loss: 1.0003
epoch: 5, train_loss: 0.9782, val_loss: 0.9984
epoch: 6, train_loss: 0.9759, val_loss: 0.9962
epoch: 7, train_loss: 0.9739, val_loss: 0.9952
epoch: 8, train_loss: 0.9724, val_loss: 0.9944
epoch: 9, train_loss: 0.9711, val_loss: 0.9932
epoch: 10, train_loss: 0.9700, val_loss: 0.9927
epoch: 11, train_loss: 0.9691, val_loss: 0.9921
epoch: 12, train_loss: 0.9684, val_loss: 0.9917
epoch: 13, train_loss: 0.9677, val_loss: 0.9913
epoch: 14, train_loss: 0.9671, val_loss: 0.9910
epoch: 15, train_loss: 0.9665, val_loss: 0.9905
epoch: 16, train_loss: 0.9659, val_loss: 0.9898
epoch: 17, train_loss: 0.9649, val_loss: 0.9890
epoch: 18, train_loss: 0.9641, val_loss: 0.9883
epoch: 19, train_loss: 0.9635, val_loss: 0.9880
epoch: 20, train_loss: 0.9630, val_loss: 0.9875
ep

## 1.2

Train the **same MLP** with three optimizers: SGD, SGD with momentum, and Adam. Keep the same architecture and hyperparameters, except for the optimizer. Plot the three validation loss curves on a single graph.

In [ ]:
reload_src()

configs = [
    {'exp_id':exp_id_1('2_1') ,'optimizer': 'sgd', 'momentum': 0.5},
    {'exp_id':exp_id_1('2_2'), 'optimizer': 'adam'}
]

for config in configs:
    print(f'Running with config: {config}')
    p1.run_1(
        p1.RunConfig(
            exp_id=config.get('exp_id'),
            device=device,
            output=p1.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
            hyperparams=p1.Hyperparams(
                epochs=EPOCHS_1,
                lr=LR_1,
                momentum=config.get('momentum'),
                hidden_sizes=HIDDEN_SIZES_1_1,
                activation=ACTIVATION_1_1,
                optimizer=config.get('optimizer'),
            ),
        )
    )

Running with config: {'optimizer': 'sgd', 'momentum': 0.5}
epoch: 0, train_loss: 1.0178, val_loss: 1.0244
epoch: 1, train_loss: 1.0005, val_loss: 1.0149
epoch: 2, train_loss: 0.9936, val_loss: 1.0106
epoch: 3, train_loss: 0.9899, val_loss: 1.0080
epoch: 4, train_loss: 0.9872, val_loss: 1.0055
epoch: 5, train_loss: 0.9847, val_loss: 1.0032
epoch: 6, train_loss: 0.9823, val_loss: 1.0009
epoch: 7, train_loss: 0.9798, val_loss: 0.9989
epoch: 8, train_loss: 0.9775, val_loss: 0.9967
epoch: 9, train_loss: 0.9752, val_loss: 0.9947
epoch: 10, train_loss: 0.9730, val_loss: 0.9927
epoch: 11, train_loss: 0.9708, val_loss: 0.9908
epoch: 12, train_loss: 0.9686, val_loss: 0.9887
epoch: 13, train_loss: 0.9663, val_loss: 0.9868
epoch: 14, train_loss: 0.9640, val_loss: 0.9847
epoch: 15, train_loss: 0.9618, val_loss: 0.9827
epoch: 16, train_loss: 0.9596, val_loss: 0.9810
epoch: 17, train_loss: 0.9573, val_loss: 0.9787
epoch: 18, train_loss: 0.9546, val_loss: 0.9765
epoch: 19, train_loss: 0.9524, val_loss

## 1.3

Start from a deep MLP with 5 hidden layers and compare the following four configurations:

| \# | Configuration |
|---|---|
| 1 | Sigmoid + default initialization |
| 2 | ReLU + He initialization |
| 3 | Configuration 2 + BatchNorm |
| 4 | Configuration 3 + Dropout ($p=0.3$) |

For each configuration, report the final validation MSE, the mean $\ell_2$ gradient norm at the first hidden layer (averaged over mini-batches of the last epoch), and the train-validation gap.

In [7]:
OPTIMIZER_1_3 = 'adam'
HIDDEN_SIZES_1_3 = [HIDDEN_SIZE_1] * 5 # 5 hidden layers

In [ ]:
reload_src()

configs = [
    { 'exp_id': exp_id_1('3_1'), 'activation': 'sigmoid' },
    { 'exp_id': exp_id_1('3_2'), 'activation': 'relu', 'initialization': 'he' },
]
configs.append({
    **configs[1],
    'batch_norm': True,
    'exp_id': exp_id_1('3_3'),
})
configs.append({
    **configs[2],
    'dropout': 0.5,
    'exp_id': exp_id_1('3_4'),
})

for config in configs:
    print(f'Running with config: {config}')
    p1.run_1(
        p1.RunConfig(
            exp_id=config.get('exp_id'),
            device=device,
            output=p1.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
            hyperparams=p1.Hyperparams(
                epochs=EPOCHS_1,
                lr=LR_1,
                momentum=config.get('momentum'),
                hidden_sizes=HIDDEN_SIZES_1_3,
                activation=config.get('activation'),
                dropout=config.get('dropout'),
                batch_norm=config.get('batch_norm'),
                optimizer=OPTIMIZER_1_3,
                initialization=config.get('initialization')
            ),
        )
    )

Running with config: {'activation': 'sigmoid'}
epoch: 0, train_loss: 1.0044, val_loss: 1.0191
epoch: 1, train_loss: 1.0004, val_loss: 1.0139
epoch: 2, train_loss: 0.9456, val_loss: 0.8864
epoch: 3, train_loss: 0.8194, val_loss: 0.7842
epoch: 4, train_loss: 0.7478, val_loss: 0.7375
epoch: 5, train_loss: 0.6966, val_loss: 0.6924
epoch: 6, train_loss: 0.6532, val_loss: 0.6577
epoch: 7, train_loss: 0.6112, val_loss: 0.6276
epoch: 8, train_loss: 0.5882, val_loss: 0.6232
epoch: 9, train_loss: 0.5571, val_loss: 0.6177
epoch: 10, train_loss: 0.5306, val_loss: 0.5929
epoch: 11, train_loss: 0.5037, val_loss: 0.6356
epoch: 12, train_loss: 0.4833, val_loss: 0.6032
epoch: 13, train_loss: 0.4682, val_loss: 0.6326
epoch: 14, train_loss: 0.4539, val_loss: 0.6426
epoch: 15, train_loss: 0.4379, val_loss: 0.6231
epoch: 16, train_loss: 0.4216, val_loss: 0.6713
epoch: 17, train_loss: 0.4147, val_loss: 0.6519
epoch: 18, train_loss: 0.3958, val_loss: 0.6598
epoch: 19, train_loss: 0.3865, val_loss: 0.6692
epo

# 2

In [9]:
import part_2 as p2

reload_src()

EPOCHS_2 = 100
OPTIMIZER_2 = 'adam'

def exp_id_2(subpart: str) -> str:
    return exp_id(f'2_{subpart}')

## 2.1

Implement the following pipeline in PyTorch:

```
SMILES → character embedding → LSTM → final state → linear layer → prediction
```

Use correct handling of variable-length sequences. If you use padding, your aggregations and masks must ignore the padding positions.

In [10]:
HIDDEN_SIZE_2_1 = 16
EMBEDDING_SIZE_2_1 = 8
DROPOUT_2_1 = 0.5
LR_2_1 = 1e-3

In [ ]:
reload_src()

p2.run_2(
    p2.RunConfig(
        exp_id=exp_id_2('1_1'),
        model='lstm',
        device=device,
        output=p2.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p2.Hyperparams(
            epochs=EPOCHS_2,
            lr=LR_2_1,
            hidden_size=HIDDEN_SIZE_2_1,
            embedding_size=EMBEDDING_SIZE_2_1,
            optimizer=OPTIMIZER_2,
            dropout=DROPOUT_2_1,
        ),
    )
)

Running with config: RunConfig(device=device(type='cpu'), output=Output(logs_dir='./logs', models_dir='./models'), hyperparams=Hyperparams(epochs=100, embedding_size=8, optimizer='adam', lr=0.001, hidden_size=16, dropout=0.5, is_bidirectional=False, num_heads=None, num_encoder_layers=None), exp_id='exp_2_1', model='lstm')


/Users/travisholt/github/trvslhlt/ift_6390_machine_learning/.venv/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


epoch: 0, train_loss: 0.9529, val_loss: 0.9284
epoch: 1, train_loss: 0.8697, val_loss: 0.8471
epoch: 2, train_loss: 0.7862, val_loss: 0.7590
epoch: 3, train_loss: 0.7224, val_loss: 0.7200
epoch: 4, train_loss: 0.6747, val_loss: 0.7068
epoch: 5, train_loss: 0.6477, val_loss: 0.6982
epoch: 6, train_loss: 0.6233, val_loss: 0.6938
epoch: 7, train_loss: 0.6007, val_loss: 0.6894
epoch: 8, train_loss: 0.5808, val_loss: 0.6968
epoch: 9, train_loss: 0.5708, val_loss: 0.6820
epoch: 10, train_loss: 0.5472, val_loss: 0.6721
epoch: 11, train_loss: 0.5248, val_loss: 0.6612
epoch: 12, train_loss: 0.5053, val_loss: 0.6494
epoch: 13, train_loss: 0.4905, val_loss: 0.6297
epoch: 14, train_loss: 0.4696, val_loss: 0.6148
epoch: 15, train_loss: 0.4438, val_loss: 0.6054
epoch: 16, train_loss: 0.4200, val_loss: 0.6019
epoch: 17, train_loss: 0.3998, val_loss: 0.5894
epoch: 18, train_loss: 0.3832, val_loss: 0.5880
epoch: 19, train_loss: 0.3639, val_loss: 0.5772
epoch: 20, train_loss: 0.3449, val_loss: 0.5830
ep

In [13]:
# try with a bidirectional LSTM
reload_src()

p2.run_2(
    p2.RunConfig(
        exp_id=exp_id_2('1_2'),
        model='lstm',
        device=device,
        output=p2.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p2.Hyperparams(
            epochs=EPOCHS_2,
            lr=LR_2_1,
            hidden_size=HIDDEN_SIZE_2_1,
            embedding_size=EMBEDDING_SIZE_2_1,
            optimizer=OPTIMIZER_2,
            dropout=DROPOUT_2_1,
            is_bidirectional=True,
        ),
    )
)

Running with config: RunConfig(device=device(type='cpu'), output=Output(logs_dir='./logs', models_dir='./models'), hyperparams=Hyperparams(epochs=100, embedding_size=8, optimizer='adam', lr=0.001, hidden_size=16, dropout=0.5, is_bidirectional=True, num_heads=None, num_encoder_layers=None), exp_id='exp_2_1_2', model='lstm')


/Users/travisholt/github/trvslhlt/ift_6390_machine_learning/.venv/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


epoch: 0, train_loss: 0.9431, val_loss: 0.8823
epoch: 1, train_loss: 0.8058, val_loss: 0.7468
epoch: 2, train_loss: 0.6924, val_loss: 0.6642
epoch: 3, train_loss: 0.6388, val_loss: 0.6265
epoch: 4, train_loss: 0.6108, val_loss: 0.6162
epoch: 5, train_loss: 0.5852, val_loss: 0.6097
epoch: 6, train_loss: 0.5619, val_loss: 0.6089
epoch: 7, train_loss: 0.5441, val_loss: 0.6002
epoch: 8, train_loss: 0.5229, val_loss: 0.5951
epoch: 9, train_loss: 0.5069, val_loss: 0.5943
epoch: 10, train_loss: 0.4857, val_loss: 0.5964
epoch: 11, train_loss: 0.4741, val_loss: 0.6028
epoch: 12, train_loss: 0.4532, val_loss: 0.6039
epoch: 13, train_loss: 0.4376, val_loss: 0.6074
epoch: 14, train_loss: 0.4223, val_loss: 0.6114
epoch: 15, train_loss: 0.4121, val_loss: 0.6189
epoch: 16, train_loss: 0.3896, val_loss: 0.6201
epoch: 17, train_loss: 0.3823, val_loss: 0.6269
epoch: 18, train_loss: 0.3706, val_loss: 0.6288
epoch: 19, train_loss: 0.3522, val_loss: 0.6330
epoch: 20, train_loss: 0.3424, val_loss: 0.6332
ep

## 2.3

Implement a transformer encoder for predicting $T_c$ with: character-by-character embedding, sinusoidal positional encoding, 2 transformer layers, mean aggregation over non-padded positions, and a final linear layer.

In [14]:
NUM_ENCODER_LAYERS_2_3 = 2
NUM_HEADS_2_3 = 4
EMBEDDING_SIZE_2_3 = 32
LR_2_3 = 1e-4
DROPOUT_2_3 = 0.1

In [ ]:
reload_src()

p2.run_2(
    p2.RunConfig(
        exp_id=exp_id_2('3'),
        model='transformer',
        device=device,
        output=p2.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p2.Hyperparams(
            epochs=EPOCHS_2,
            lr=LR_2_3,
            embedding_size=EMBEDDING_SIZE_2_3,
            optimizer=OPTIMIZER_2,
            dropout=DROPOUT_2_3,
            num_encoder_layers=NUM_ENCODER_LAYERS_2_3,
            num_heads=NUM_HEADS_2_3,
        ),
    )
)

Running with config: RunConfig(device=device(type='cpu'), output=Output(logs_dir='./logs', models_dir='./models'), hyperparams=Hyperparams(epochs=25, embedding_size=32, optimizer='adam', lr=0.0001, hidden_size=None, dropout=0.1, is_bidirectional=False, num_heads=4, num_encoder_layers=2), exp_id='exp_2_3', model='transformer')


/Users/travisholt/github/trvslhlt/ift_6390_machine_learning/assignments/assignment_2/./src/data_prep.py:127: DtypeWarning: Columns (0,6,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df.columns = ['smiles', 'target']


epoch: 0, train_loss: 0.8630, val_loss: 0.7867
epoch: 1, train_loss: 0.7610, val_loss: 0.7428
epoch: 2, train_loss: 0.7236, val_loss: 0.7216
epoch: 3, train_loss: 0.7005, val_loss: 0.6908
epoch: 4, train_loss: 0.6844, val_loss: 0.6684
epoch: 5, train_loss: 0.6690, val_loss: 0.6584
epoch: 6, train_loss: 0.6601, val_loss: 0.6482
epoch: 7, train_loss: 0.6506, val_loss: 0.6446
epoch: 8, train_loss: 0.6328, val_loss: 0.6591
epoch: 9, train_loss: 0.6344, val_loss: 0.6384
epoch: 10, train_loss: 0.6196, val_loss: 0.6292
epoch: 11, train_loss: 0.6080, val_loss: 0.6237
epoch: 12, train_loss: 0.5882, val_loss: 0.6669
epoch: 13, train_loss: 0.5725, val_loss: 0.6361
epoch: 14, train_loss: 0.5778, val_loss: 0.6306
epoch: 15, train_loss: 0.5462, val_loss: 0.6118
epoch: 16, train_loss: 0.5328, val_loss: 0.6267
epoch: 17, train_loss: 0.5178, val_loss: 0.6025
epoch: 18, train_loss: 0.5106, val_loss: 0.6531
epoch: 19, train_loss: 0.4919, val_loss: 0.6042
epoch: 20, train_loss: 0.4823, val_loss: 0.6243
ep